In [1]:
############## Code to fetch text line info from a single  normal pdf #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json

input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_12_09_2025.pdf"
# input_pdf_path =r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
# input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_21_08_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_24_09_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_26_08_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
reader = PdfReader(input_pdf_path)
writer = PdfWriter()

# -----------------------------
# Extract all lines from PDF
# -----------------------------
all_lines = []

with pdfplumber.open(input_pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        # print("page_num", page_num)
        for line in page.extract_text_lines():
            # print(line['text'])
            all_lines.append(line['text'].replace("\xa0", " ").strip())



# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------
company_info = {}
auditors = []
i = 0
while i < len(all_lines):
    line = all_lines[i]
    # print(line)
    # print("*Corporate Identity Number (CIN)" in line)
    if '*Corporate Identity Number (CIN)' in line:
        company_info['cin'] = all_lines[i + 1].strip()
    elif '*Name of the company' in line:
        name_parts = []
        j = i + 1
        while j < len(all_lines) and not all_lines[j].startswith('(b)'):
            name_parts.append(all_lines[j].strip())
            j += 1
        company_info['name'] = ' '.join(name_parts)
    elif '*Address of the registered office of the company' in line:
        addr_parts = []
        j = i + 1
        while j < len(all_lines) and not all_lines[j].startswith('(c)'):
            addr_parts.append(all_lines[j].strip())
            j += 1
        company_info['address'] = ', '.join(addr_parts)
    elif 'e-mail ID' in line:
        company_info['email'] = all_lines[i + 1].strip()
    elif '*From (DD/MM/YYYY)' in line:
        # print("coming here")
        company_info['Financial Year data from'] = all_lines[i + 1].strip()
    elif '*To (DD/MM/YYYY)' in line:
        # print("coming here")
        company_info['Financial Year data to'] = all_lines[i + 1].strip()
    elif "Date of AGM (DD/MM/YYYY)".lower() in line.lower() and "due" not in line.lower():
        company_info['DateOnWhichAGMIsHeld'] = all_lines[i + 1].strip()
    elif 'Nature of financial statements'.lower() in line.lower():
        # print("coming here")
        company_info['ReportNature'] = all_lines[i + 1].strip()
    elif "*Number of Auditors".lower() in line.lower():
        number_of_auditors = all_lines[i + 1].strip().split(" ")[-1]
        company_info['number_of_auditors'] = all_lines[i + 1].strip().split(" ")[-1]
        print("coming here", all_lines[i + 1].strip().split(" ")[-1])

    elif 'Income-tax (PAN) of auditor or auditor’s firm'.lower() in line.lower():
        # print("coming here")
        auditor = {}
        auditor['AuditorFirmPAN'] = all_lines[i + 1].strip()
    elif 'Name of the auditor or auditor’s firm'.lower() in line.lower():
        # print("coming here")
        auditor['NameOfAuditorFirm'] = all_lines[i + 1].strip()
    elif 'Membership number of auditor or auditor’s firm’s registration'.lower() in line.lower():
        # print("coming here")
        auditor['AuditFirmRegNum'] = all_lines[i + 1].strip()
    elif 'Address of the auditor or auditor’s firm'.lower() in line.lower():
        # print("coming here")
        address = ""
        address_line_1 = ""
        address_line_2 = ""
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "Address Line 1".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                # address = all_lines[i+j+1].strip()
                addr1_parts = []
                j = i + 2
                while j < len(all_lines) and "address line 2" not in all_lines[j].lower():
                    addr1_parts.append(all_lines[j].strip())
                    j += 1
                address = ', '.join(addr1_parts)
                break
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "Address Line 2".lower() in all_lines[i+j].lower():
                print("coming here", all_lines[i+j+1].strip())
                if "country" not in all_lines[i+j+1].lower():
                    address_line_2 = all_lines[i+j+1].strip()
                    address = address + " " + address_line_2
                else:
                    break
        auditor['AddressOfAuditor'] = address
        # print(all_lines[i+1])
        # if "Address Line 1".lower() in all_lines[i+1].lower():
        #     address = all_lines[i + 2].strip()
        # if "Address Line 2" in all_lines[i+3]:
        #     address_line_2 = all_lines[i + 4].strip()
        #     if "country" not in address_line_2.lower():
        #         address = address + " " + address_line_2
        # company_info['AddressOfAuditor1'] = address.strip()
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "Country".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                auditor['AuditorAddressCountry'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "pin code".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                auditor['AuditorAddressPIN'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 15):
            # print("coming here", all_lines[i+j].lower())
            if "city".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                auditor['AuditorAddressCity'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 15):
            # print("coming here", all_lines[i+j].lower())
            if "state".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                auditor['AuditorAddressState'] = all_lines[i+j+1].strip()
                break
    elif 'Details of the member signing for the above firm'.lower() in line.lower():
        for j in range(1, 5):
            # print("coming here", all_lines[i+j].lower())
            if "Name of the member".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                auditor['AuditorMemberName'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 5):
            # print("coming here", all_lines[i+j].lower())
            if "Membership number".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                auditor['AuditorMemberNumber'] = all_lines[i+j+1].strip()
                break        
        auditors.append(auditor)
    i += 1
company_info['auditors'] = auditors
# -----------------------------
# :two: Extract financial data
# -----------------------------
financial_data = []
i = 0
while i < len(all_lines):
    line = all_lines[i]
    if not is_number(line):  # heading lines
        heading_parts = [line]
        j = i + 1
        while j < len(all_lines) and not is_number(all_lines[j]):
            heading_parts.append(all_lines[j])
            j += 1
        heading = ' '.join(heading_parts)
        # Collect two numeric values if available
        if j + 1 < len(all_lines) and is_number(all_lines[j]) and is_number(all_lines[j + 1]):
            values = [float(all_lines[j].replace(',', '')), float(all_lines[j + 1].replace(',', ''))]
            financial_data.append({heading: values})
            i = j + 2
        else:
            i += 1
    else:
        i += 1
# -----------------------------
# :three: Combine and output JSON
# -----------------------------
output = {
    "company_info": company_info,
    "financial_data": financial_data
}
json_output = json.dumps(output, indent=4)
output['company_info']


coming here 2
coming here Pherozeshah Mehta Road
coming here Chakala, Andheri East


{'cin': 'U99999MH1917PTC000478',
 'name': 'TATA SONS PRIVATE LIMITED',
 'address': 'BOMBAY HOUSE 24 HOMI MODY, STREET, FORT, NA, MUMBAI,, Maharashtra, India, 400001.',
 'email': '*****kash.mukhopadhyay@tata.co',
 'Financial Year data from': '01/04/2024',
 'Financial Year data to': '31/03/2025',
 'ReportNature': 'Adopted Financial statements',
 'DateOnWhichAGMIsHeld': '14/08/2025',
 'number_of_auditors': '2',
 'auditors': [{'AuditorFirmPAN': 'AAAFN4217G',
   'AuditFirmRegNum': '108296W',
   'NameOfAuditorFirm': 'N M Raiji & Co.',
   'AddressOfAuditor': 'Universal Insurance Building, Pherozeshah Mehta Road',
   'AuditorAddressCountry': 'India',
   'AuditorAddressPIN': '400001',
   'AuditorAddressCity': 'Mumbai',
   'AuditorAddressState': 'Maharashtra',
   'AuditorMemberName': 'Vinay D. Balse',
   'AuditorMemberNumber': '039434'},
  {'AuditorFirmPAN': 'AAAFB1386J',
   'AuditFirmRegNum': '101490W',
   'NameOfAuditorFirm': 'Bilimoria Mehta & Co',
   'AddressOfAuditor': '507/508, 5th Floor I

In [1]:
############## Code to fetch text line info from a multiple pdfs #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import os

# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------


input_dir = "pdfs"

for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        print("filename :", filename)
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
        # input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
        input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\" + filename
        # reader = PdfReader(input_pdf_path)
        # writer = PdfWriter()

        # -----------------------------
        # Extract all lines from PDF
        # -----------------------------
        all_lines = []

        with pdfplumber.open(input_pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # print("page_num", page_num)
                for line in page.extract_text_lines():
                    # print(line['text'])
                    if "Part B - Balance Sheet" in line['text']:
                        print("starting page of balance sheet", page_num + 1)
                        # print(line['text'])
                        BS_starting_page = page_num + 1
                    if "TOTAL LIABILITIES" in line['text']:
                        print("ending page of balance sheet", page_num+1)
                        BS_ending_page = page_num + 1
                    all_lines.append(line['text'].replace("\xa0", " ").strip())




        company_info = {}
        i = 0
        while i < len(all_lines):
            line = all_lines[i]
            # print(line)
            # print("*Corporate Identity Number (CIN)" in line)
            if '*Corporate Identity Number (CIN)' in line:
                company_info['cin'] = all_lines[i + 1].strip()
            elif '*Name of the company' in line:
                name_parts = []
                j = i + 1
                while j < len(all_lines) and not all_lines[j].startswith('(b)'):
                    name_parts.append(all_lines[j].strip())
                    j += 1
                company_info['name'] = ' '.join(name_parts)
            elif '*Address of the registered office of the company' in line:
                addr_parts = []
                j = i + 1
                while j < len(all_lines) and not all_lines[j].startswith('(c)'):
                    addr_parts.append(all_lines[j].strip())
                    j += 1
                company_info['address'] = ', '.join(addr_parts)
            elif 'e-mail ID' in line:
                company_info['email'] = all_lines[i + 1].strip()
            elif '*From (DD/MM/YYYY)' in line:
                # print("coming here")
                company_info['Financial Year data from'] = all_lines[i + 1].strip()
            elif '*To (DD/MM/YYYY)' in line:
                # print("coming here")
                company_info['Financial Year data to'] = all_lines[i + 1].strip()
            elif "Date of AGM (DD/MM/YYYY)".lower() in line.lower() and "due" not in line.lower():
                company_info['DateOnWhichAGMIsHeld'] = all_lines[i + 1].strip()
            elif 'Nature of financial statements'.lower() in line.lower():
                # print("coming here")
                company_info['ReportNature'] = all_lines[i + 1].strip()
            elif 'Income-tax (PAN) of auditor or auditor’s firm'.lower() in line.lower():
                # print("coming here")
                company_info['AuditorFirmPAN'] = all_lines[i + 1].strip()
            elif 'Name of the auditor or auditor’s firm'.lower() in line.lower():
                # print("coming here")
                company_info['NameOfAuditorFirm'] = all_lines[i + 1].strip()
            elif 'Membership number of auditor or auditor’s firm’s registration'.lower() in line.lower():
                # print("coming here")
                company_info['AuditFirmRegNum'] = all_lines[i + 1].strip()
            elif 'Address of the auditor or auditor’s firm'.lower() in line.lower():
                # print("coming here")
                address = ""
                address_line_1 = ""
                address_line_2 = ""
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "Address Line 1".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        address = all_lines[i+j+1].strip()
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "Address Line 2".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        if "country" not in all_lines[i+j+1].lower():
                            address_line_2 = all_lines[i+j+1].strip()
                            address = address + " " + address_line_2
                        break
                company_info['AddressOfAuditor1'] = address
                # print(all_lines[i+1])
                # if "Address Line 1".lower() in all_lines[i+1].lower():
                #     address = all_lines[i + 2].strip()
                # if "Address Line 2" in all_lines[i+3]:
                #     address_line_2 = all_lines[i + 4].strip()
                #     if "country" not in address_line_2.lower():
                #         address = address + " " + address_line_2
                # company_info['AddressOfAuditor1'] = address.strip()
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "Country".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['AuditorAddressCountry'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "pin code".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['AuditorAddressPIN'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "city".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['AuditorAddressCity'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "state".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['AuditorAddressState'] = all_lines[i+j+1].strip()
                        break
            elif 'Details of the member signing for the above firm'.lower() in line.lower():
                for j in range(1, 5):
                    # print("coming here", all_lines[i+j].lower())
                    if "Name of the member".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['AuditorMemberName'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 5):
                    # print("coming here", all_lines[i+j].lower())
                    if "Membership number".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['AuditorMemberNumber'] = all_lines[i+j+1].strip()
                        break

                
                

                
                
                

            i += 1
        # -----------------------------
        # :two: Extract financial data
        # -----------------------------
        financial_data = []
        i = 0
        while i < len(all_lines):
            line = all_lines[i]
            if not is_number(line):  # heading lines
                heading_parts = [line]
                j = i + 1
                while j < len(all_lines) and not is_number(all_lines[j]):
                    heading_parts.append(all_lines[j])
                    j += 1
                heading = ' '.join(heading_parts)
                # Collect two numeric values if available
                if j + 1 < len(all_lines) and is_number(all_lines[j]) and is_number(all_lines[j + 1]):
                    values = [float(all_lines[j].replace(',', '')), float(all_lines[j + 1].replace(',', ''))]
                    financial_data.append({heading: values})
                    i = j + 2
                else:
                    i += 1
            else:
                i += 1
        # -----------------------------
        # :three: Combine and output JSON
        # -----------------------------
        output = {
            "company_info": company_info,
            "financial_data": financial_data
        }
        json_output = json.dumps(output, indent=4)
        # print(output['company_info'])
        for key, value in output['company_info'].items():
            print(f"{key}: {value}")

        print("----------------------------------------------------------------")
        


filename : AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf
starting page of balance sheet 6
ending page of balance sheet 9
cin: L67120MH1993PLC070739
name: UGRO CAPITAL LIMITED
address: Equinox Business Park, Tower 3,, Fourth Floor, Off BKC, LBS Road,, Kurla Mumbai - 400070, NA,, Mumbai, Mumbai City,, Maharashtra, India, 400070.
email: *****rocapital.com
Financial Year data from: 01/04/2024
Financial Year data to: 31/03/2025
ReportNature: Adopted Financial statements
DateOnWhichAGMIsHeld: 08/08/2025
AuditorFirmPAN: AAAFS1034J
AuditFirmRegNum: 109983W
NameOfAuditorFirm: Sharp & Tannan Associates
AddressOfAuditor1: 87 Nariman Bhavan,
AuditorAddressCountry: India
AuditorAddressPIN: 400021
AuditorAddressCity: Mumbai
AuditorMemberName: Tirtharaj Khot
AuditorMemberNumber: 037457
----------------------------------------------------------------
filename : AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025.pdf
starting page of balance sheet 7
ending page of balance sheet 11
cin: U65191TN1994PLC07

In [1]:
############## Code to get balance sheet and PnL data for normal pdfs table data #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import os

# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------


input_dir = "pdfs"

for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        print("filename :", filename)
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
        # input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
        input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\" + filename
        # reader = PdfReader(input_pdf_path)
        # writer = PdfWriter()

        # -----------------------------
        # Extract all lines from PDF
        # -----------------------------
        all_lines = []
        with pdfplumber.open(input_pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # print("page_num", page_num)
                for line in page.extract_text_lines():
                    page_line_list = [line['text'] for line in page.extract_text_lines()]
                    # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))

                    ######################### For getting Balance sheet page numbers ###########################################
                    if "Part B - Balance Sheet" in line['text'] or "Consolidated Balance Sheet" in line['text']:
                        if any("Figures as at the".lower() in item.lower() for item in page_line_list):
                            print("starting page of balance sheet", page_num + 1)
                            # page_line_list = [line['text'] for line in page.extract_text_lines()]
                            # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))
                            BS_starting_page = page_num + 1
                        else:
                            BS_starting_page = page_num + 2
                            print("starting page of balance sheet", page_num + 2)
                    if "TOTAL LIABILITIES" in line['text']:
                        print("ending page of balance sheet", page_num+1)
                        BS_ending_page = page_num +1 


                    ##################### For getting profit and loss page numbers ##########################################
                    if "Statement of Profit and Loss".lower() in line['text'].lower() or "Statement of Consolidated Profit and Loss".lower() in line['text'].lower():
                        if any("Figures for the period".lower() in item.lower() for item in page_line_list):
                            print("starting page of profit and loss", page_num + 1)
                            # page_line_list = [line['text'] for line in page.extract_text_lines()]
                            # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))
                            PnL_starting_page = page_num + 1
                        else:
                            PnL_starting_page = page_num + 2
                            print("starting page of profit and loss", page_num + 2)
                    # if "Financial parameters - Profit and loss".lower() in line['text'].lower():
                    if "Earnings per equity share (for continuing and discontinuing operations)".lower() in line['text'].lower():  
                        print("ending page of profit and loss", page_num+1)
                        PnL_ending_page = page_num +1    

        # all_rows = []
        with pdfplumber.open(input_pdf_path) as pdf:
            all_rows_BS = []
            all_rows_PnL = []
            for page_num, page in enumerate(pdf.pages):
                if page_num + 1 >= BS_starting_page and page_num + 1 <=BS_ending_page: 
                    tables = page.extract_tables()
                    print("page_num :", page_num + 1, len(tables))
                    for table in tables:
                        # print(len(table[0]), table[0])
                        if page_num +1 == BS_starting_page: 
                            BS_header = table[0]
                            for t in table[1:]:
                                if len(t) == 7:
                                    print(len(t), t)
                                    # all_rows.append({
                                    #     "Particulars" : t[1],
                                    #     "Figures as at the\nend of (Current\nreporting period)\n(in Rs.)\n31/03/2025\n(DD/MM/YYYY)" : t[2],
                                    #     "Figures as at the\nend of (Previous\nreporting period)\n(in Rs.)\n31/03/2024\n(DD/MM/YYYY)" : t[3],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod" : t[4],
                                    #     "Figures as at the\nbeginning of\n(Previous reporting\nperiod) (in Rs.)\n(DD/MM/YYYY)" : t[5],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod"  : t[6],
                                    #     })
                                    all_rows_BS.append({
                                        BS_header[1] : t[1],
                                        BS_header[2] : t[2],
                                        BS_header[3] : t[3],
                                        # BS_header[4] : t[4],
                                        # BS_header[5] : t[5],
                                        # BS_header[6]  : t[6],
                                        })
                        else:
                            for t in table:
                                if len(t) == 7:
                                    print(len(t), t)
                                    # all_rows.append({
                                    #     "Particulars" : t[1],
                                    #     "Figures as at the\nend of (Current\nreporting period)\n(in Rs.)\n31/03/2025\n(DD/MM/YYYY)" : t[2],
                                    #     "Figures as at the\nend of (Previous\nreporting period)\n(in Rs.)\n31/03/2024\n(DD/MM/YYYY)" : t[3],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod" : t[4],
                                    #     "Figures as at the\nbeginning of\n(Previous reporting\nperiod) (in Rs.)\n(DD/MM/YYYY)" : t[5],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod"  : t[6],
                                    #     })
                                    all_rows_BS.append({
                                        BS_header[1] : t[1],
                                        BS_header[2] : t[2],
                                        BS_header[3] : t[3],
                                        # BS_header[4] : t[4],
                                        # BS_header[5] : t[5],
                                        # BS_header[6]  : t[6],
                                        })
                

                if page_num + 1 >= PnL_starting_page and page_num + 1 <=PnL_ending_page: 
                    tables = page.extract_tables()
                    print("page_num :", page_num + 1, len(tables))
                    if page_num + 1 == PnL_starting_page and len(tables) >=2:
                        tables = tables[len(tables)-1:]
                    if page_num + 1 == PnL_ending_page and len(tables) >=2:
                        tables = tables[:1]
                    for table in tables:
                        # print(len(table[0]), table[0])
                        if page_num +1 == PnL_starting_page: 
                            PnL_header = table[0]
                            for t in table[1:]:
                                if len(t) == 5:
                                    print(len(t), t)
                                    all_rows_PnL.append({
                                        PnL_header[1] : t[1],
                                        PnL_header[2] : t[2],
                                        PnL_header[3] : t[3],
                                        # PnL_header[4] : t[4],
                                        # PnL_header[5] : t[5],
                                        # PnL_header[6]  : t[6],
                                        })
                        else:
                            for t in table:
                                if len(t) == 5:
                                    print(len(t), t)
                                    all_rows_PnL.append({
                                        PnL_header[1] : t[1],
                                        PnL_header[2] : t[2],
                                        PnL_header[3] : t[3],
                                        # PnL_header[4] : t[4],
                                        # PnL_header[5] : t[5],
                                        # PnL_header[6]  : t[6],
                                        })

filename : AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf
starting page of balance sheet 6
ending page of balance sheet 9
starting page of profit and loss 19
ending page of profit and loss 22
page_num : 6 1
7 ['I', 'ASSETS', None, None, None, None, None]
7 ['(1)', 'Financial Assets', None, None, None, None, None]
7 ['', '(a) Cash and cash\nequivalents', '1892419353.50', '883515493.13', '', '', '']
7 ['', '(b) Bank Balance\nother than\n(a) above', '3551531662.53', '3665291193.94', '', '', '']
7 ['', '(c) Derivative\nfinancial instruments', '186121020.38', '0', '', '', '']
7 ['', '(d) Receivables', None, None, None, None, None]
7 ['', '(I) Trade Receivables', '0', '0', '', '', '']
7 ['', '(II) Other Receivables', '214196845.01', '74108528.79', '', '', '']
7 ['', 'e) Loans', '79191094842.67', '54322103008.51', '', '', '']
7 ['', '(f) Investments', '1034031323.27', '591860045.82', '', '', '']
7 ['', '(g) Other Financial\nAssets', '159970531.26', '128010554.56', '', '', '']
7 ['(2)',

In [4]:
############## Code for CFs type of pdfs table data #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import os

# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------


input_dir = "pdfs\\CFS"

for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        print("filename :", filename)
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
        # input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
        input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\CFS\\" + filename
        # reader = PdfReader(input_pdf_path)
        # writer = PdfWriter()

        # -----------------------------
        # Extract all lines from PDF
        # -----------------------------
        all_lines = []
        with pdfplumber.open(input_pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # print("page_num", page_num)
                for line in page.extract_text_lines():
                    page_line_list = [line['text'] for line in page.extract_text_lines()]
                    # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))

                    ######################### For getting Balance sheet page numbers ###########################################
                    if "Part B - Balance Sheet" in line['text'] or "Consolidated Balance Sheet" in line['text']:
                        if any("Figures as at the".lower() in item.lower() for item in page_line_list):
                            print("starting page of balance sheet", page_num + 1)
                            # page_line_list = [line['text'] for line in page.extract_text_lines()]
                            # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))
                            BS_starting_page = page_num + 1
                        else:
                            BS_starting_page = page_num + 2
                            print("starting page of balance sheet", page_num + 2)
                    if "TOTAL LIABILITIES" in line['text']:
                        print("ending page of balance sheet", page_num+1)
                        BS_ending_page = page_num +1 


                    ##################### For getting profit and loss page numbers ##########################################
                    if "Statement of Profit and Loss".lower() in line['text'].lower() or "Statement of Consolidated Profit and Loss".lower() in line['text'].lower():
                        if any("Figures for the period".lower() in item.lower() for item in page_line_list):
                            print("starting page of profit and loss", page_num + 1)
                            # page_line_list = [line['text'] for line in page.extract_text_lines()]
                            # print(any("Figures as at the".lower() in item.lower() for item in page_line_list))
                            PnL_starting_page = page_num + 1
                        else:
                            PnL_starting_page = page_num + 2
                            print("starting page of profit and loss", page_num + 2)
                    # if "Financial parameters - Profit and loss".lower() in line['text'].lower():
                    if "Earnings per equity share (for continuing and discontinuing operations)".lower() in line['text'].lower():  
                        print("ending page of profit and loss", page_num+1)
                        PnL_ending_page = page_num +1    

        # all_rows = []
        with pdfplumber.open(input_pdf_path) as pdf:
            all_rows_BS = []
            all_rows_PnL = []
            for page_num, page in enumerate(pdf.pages):
                if page_num + 1 >= BS_starting_page and page_num + 1 <=BS_ending_page: 
                    tables = page.extract_tables()
                    print("page_num :", page_num + 1, len(tables))
                    for table in tables:
                        # print(len(table[0]), table[0])
                        if page_num +1 == BS_starting_page: 
                            BS_header = table[0]
                            for t in table[1:]:
                                if len(t) == 7:
                                    print(len(t), t)
                                    # all_rows.append({
                                    #     "Particulars" : t[1],
                                    #     "Figures as at the\nend of (Current\nreporting period)\n(in Rs.)\n31/03/2025\n(DD/MM/YYYY)" : t[2],
                                    #     "Figures as at the\nend of (Previous\nreporting period)\n(in Rs.)\n31/03/2024\n(DD/MM/YYYY)" : t[3],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod" : t[4],
                                    #     "Figures as at the\nbeginning of\n(Previous reporting\nperiod) (in Rs.)\n(DD/MM/YYYY)" : t[5],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod"  : t[6],
                                    #     })
                                    all_rows_BS.append({
                                        BS_header[1] : t[1],
                                        BS_header[2] : t[2],
                                        BS_header[3] : t[3],
                                        # BS_header[4] : t[4],
                                        # BS_header[5] : t[5],
                                        # BS_header[6]  : t[6],
                                        })
                        else:
                            for t in table:
                                if len(t) == 7:
                                    print(len(t), t)
                                    # all_rows.append({
                                    #     "Particulars" : t[1],
                                    #     "Figures as at the\nend of (Current\nreporting period)\n(in Rs.)\n31/03/2025\n(DD/MM/YYYY)" : t[2],
                                    #     "Figures as at the\nend of (Previous\nreporting period)\n(in Rs.)\n31/03/2024\n(DD/MM/YYYY)" : t[3],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod" : t[4],
                                    #     "Figures as at the\nbeginning of\n(Previous reporting\nperiod) (in Rs.)\n(DD/MM/YYYY)" : t[5],
                                    #     "Reason for\nchange in\npre-filled\nfigures of\nprevious\nreporting\nperiod"  : t[6],
                                    #     })
                                    all_rows_BS.append({
                                        BS_header[1] : t[1],
                                        BS_header[2] : t[2],
                                        BS_header[3] : t[3],
                                        # BS_header[4] : t[4],
                                        # BS_header[5] : t[5],
                                        # BS_header[6]  : t[6],
                                        })
                

                if page_num + 1 >= PnL_starting_page and page_num + 1 <=PnL_ending_page: 
                    tables = page.extract_tables()
                    print("page_num :", page_num + 1, len(tables))
                    if page_num + 1 == PnL_starting_page and len(tables) >=2:
                        tables = tables[len(tables)-1:]
                    if page_num + 1 == PnL_ending_page and len(tables) >=2:
                        tables = tables[:1]
                    for table in tables:
                        # print(len(table[0]), table[0])
                        if page_num +1 == PnL_starting_page: 
                            PnL_header = table[1]
                            for t in table[2:]:
                                # print("coming here", len(t), t[0])
                                if len(t) == 7:
                                    print(len(t), t)
                                    all_rows_PnL.append({
                                        PnL_header[2] : t[2],
                                        PnL_header[3] : t[3],
                                        PnL_header[4] : t[4],
                                        # PnL_header[4] : t[4],
                                        # PnL_header[5] : t[5],
                                        # PnL_header[6]  : t[6],
                                        })
                        else:
                            # print("tables in page :", len(table), table)
                            for t in table:
                                # print("coming here", len(t), t[0])
                                if len(t) == 7:
                                    print(len(t), t)
                                    all_rows_PnL.append({
                                        PnL_header[2] : t[2],
                                        PnL_header[3] : t[3],
                                        PnL_header[4] : t[4],
                                        # PnL_header[4] : t[4],
                                        # PnL_header[5] : t[5],
                                        # PnL_header[6]  : t[6],
                                        })

                                    # if t[0] is None or t[0] == '':
                                    #     print(len(t), t)
                                    #     all_rows_PnL.append({
                                    #         PnL_header[2] : t[2],
                                    #         PnL_header[3] : t[3],
                                    #         PnL_header[4] : t[4],
                                    #         # PnL_header[4] : t[4],
                                    #         # PnL_header[5] : t[5],
                                    #         # PnL_header[6]  : t[6],
                                    #         })
                                    # else:
                                    #     print(len(t), t)
                                    #     all_rows_PnL.append({
                                    #         PnL_header[2] : t[2],
                                    #         PnL_header[3] : t[3],
                                    #         PnL_header[4] : t[4],
                                    #         # PnL_header[4] : t[4],
                                    #         # PnL_header[5] : t[5],
                                    #         # PnL_header[6]  : t[6],
                                    #         })

filename : AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf
starting page of balance sheet 4
ending page of balance sheet 6
starting page of profit and loss 16
ending page of profit and loss 18
page_num : 4 1
7 ['I', 'ASSETS', None, None, None, None, None]
7 ['(1)', 'Financial Assets', None, None, None, None, None]
7 ['', '(a) Cash and cash\nequivalents', '20666306143', '24698665291', '', '', '']
7 ['', '(b) Bank Balance\nother than (a)\nabove', '21251466904', '17758507272', '', '', '']
7 ['', '(c) Derivative\nfinancial\ninstruments', '502497204', '1576863135', '', '', '']
7 ['', '(d) Receivables', None, None, None, None, None]
7 ['', '(I) Trade\nReceivables', '1075259665', '1024228512', '', '', '']
7 ['', '(II) Other\nReceivables', '0', '296523275', '', '', '']
7 ['', 'e) Loans', '553642574647', '509523153683', '', '', '']
7 ['', '(f) Investments', '44379871563', '40589828580', '', '', '']
7 ['', '(g) Other Financial\nAssets', '11429746262', '14207706911', 'Regrouping for\nbett

In [ ]:
########################## Code for CFS NBFC for PnL Data from single pdf ##############################################################
input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
PnL_starting_page = 16
PnL_ending_page = 18
with pdfplumber.open(input_pdf_path) as pdf:
    all_rows_PnL = []
    for page_num, page in enumerate(pdf.pages):
        if page_num + 1 >= PnL_starting_page and page_num + 1 <=PnL_ending_page: 
            tables = page.extract_tables()
            print("page_num :", page_num + 1, len(tables))
            if page_num + 1 == PnL_starting_page and len(tables) >=2:
                tables = tables[len(tables)-1:]
            if page_num + 1 == PnL_ending_page and len(tables) >=2:
                tables = tables[:1]
            for table in tables:
                # print(len(table[0]), table[0])
                if page_num +1 == PnL_starting_page: 
                    PnL_header = table[1]
                    for t in table[2:]:
                        # print("coming here", len(t), t[0])
                        if len(t) == 7:
                            print(len(t), t)
                            all_rows_PnL.append({
                                PnL_header[2] : t[2],
                                PnL_header[3] : t[3],
                                PnL_header[4] : t[4],
                                # PnL_header[4] : t[4],
                                # PnL_header[5] : t[5],
                                # PnL_header[6]  : t[6],
                                })
                else:
                    # print("tables in page :", len(table), table)
                    for t in table:
                        # print("coming here", len(t), t[0])
                        if len(t) == 7:
                            print(len(t), t)
                            all_rows_PnL.append({
                                PnL_header[2] : t[2],
                                PnL_header[3] : t[3],
                                PnL_header[4] : t[4],
                                # PnL_header[4] : t[4],
                                # PnL_header[5] : t[5],
                                # PnL_header[6]  : t[6],
                                })

                            # if t[0] is None or t[0] == '':
                            #     print(len(t), t)
                            #     all_rows_PnL.append({
                            #         PnL_header[2] : t[2],
                            #         PnL_header[3] : t[3],
                            #         PnL_header[4] : t[4],
                            #         # PnL_header[4] : t[4],
                            #         # PnL_header[5] : t[5],
                            #         # PnL_header[6]  : t[6],
                            #         })
                            # else:
                            #     print(len(t), t)
                            #     all_rows_PnL.append({
                            #         PnL_header[2] : t[2],
                            #         PnL_header[3] : t[3],
                            #         PnL_header[4] : t[4],
                            #         # PnL_header[4] : t[4],
                            #         # PnL_header[5] : t[5],
                            #         # PnL_header[6]  : t[6],
                            #         })

page_num : 16 1
7 [None, '', 'Revenue from operations', None, None, None, None]
7 [None, '(i)', 'Interest Income', '95042901043', '98386294047', '', None]
7 [None, '(ii)', 'Dividend Income', '21921093', '579412', '', None]
7 [None, '(iii)', 'Rental Income', '0', '0', '', None]
7 ['', '(iv)', 'Fees and commission\nIncome', '4856394816', '4110747502', 'Regrouping for better\npresentation', '']
7 [None, '(v)', 'Net gain on fair value\nchanges', '2187768754', '0', '', None]
7 [None, '(vi)', 'Net gain on derecognition of\nfinancial instruments under\namortised cost category', '0', '0', '', None]
7 [None, '(vii)', 'Sale of products (including\nExcise Duty)', '0', '0', '', None]
7 [None, '(viii)', 'Sale of services', '0', '0', '', None]
7 [None, '(ix)', 'Others', '0', '0', '', None]
7 [None, '(I)', 'Total Revenue from\noperations', '102108985706.00', '102497620961.00', '', None]
7 [None, '(II)', 'Other Income', '261698683', '2407078949', 'Regrouping for better\npresentation', None]
7 [None, '

In [6]:
# Step 3: Convert all rows to DataFrame (reset index safely)
import pandas as pd
if all_rows_BS:
    BS_df = pd.DataFrame(all_rows_BS).reset_index(drop=True)
    PnL_df = pd.DataFrame(all_rows_PnL).reset_index(drop=True)
    print("✅ Extracted rows BS:", len(BS_df))
    print("✅ Extracted rows PnL:", len(PnL_df))

    # Optional: Save output for each file
    output_path = os.path.join(input_dir, f"BalanceSheet.xlsx")
    BS_df.to_excel(output_path, index=False)
    print("💾 Saved:", output_path)

    # Optional: Save output for each file
    output_path = os.path.join(input_dir, f"PnL.xlsx")
    PnL_df.to_excel(output_path, index=False)
    print("💾 Saved:", output_path)
else:
    print("⚠️ No valid table rows found in", filename)

✅ Extracted rows BS: 52
✅ Extracted rows PnL: 66
💾 Saved: pdfs\CFS\BalanceSheet.xlsx
💾 Saved: pdfs\CFS\PnL.xlsx


In [ ]:
############################## Old Code of Shivam ####################################################
import fitz  # PyMuPDF
import json
# -----------------------------
# PDF path
# -----------------------------
# pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
# pdf_path = r"D:\nitin\AOC 25\AOC-4_Form AOC4_22_08_2025.pdf"
doc = fitz.open(pdf_path)
# -----------------------------
# Extract all lines from PDF
# -----------------------------
all_lines = []
for page in doc:
    text = page.get_text("text")
    if text:
        text = text.replace("\xa0", " ")  # normalize spaces
        lines = text.split("\n")
        all_lines.extend([line.strip() for line in lines if line.strip()])
# print(all_lines)
# -----------------------------
# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------
company_info = {}
i = 0
while i < len(all_lines):
    line = all_lines[i]
    # print(line)
    # print("*Corporate Identity Number (CIN)" in line)
    if '*Corporate Identity Number (CIN)' in line:
        company_info['cin'] = all_lines[i + 1].strip()
    elif '*Name of the company' in line:
        name_parts = []
        j = i + 1
        while j < len(all_lines) and not all_lines[j].startswith('(b)'):
            name_parts.append(all_lines[j].strip())
            j += 1
        company_info['name'] = ' '.join(name_parts)
    elif '*Address of the registered office of the company' in line:
        addr_parts = []
        j = i + 1
        while j < len(all_lines) and not all_lines[j].startswith('(c)'):
            addr_parts.append(all_lines[j].strip())
            j += 1
        company_info['address'] = ', '.join(addr_parts)
    elif '*e-mail ID of  the company' in line:
        company_info['email'] = all_lines[i + 1].strip()
    elif '*From (DD/MM/YYYY)' in line:
        # print("coming here")
        company_info['Financial Year data from'] = all_lines[i + 2].strip()
    elif '*To (DD/MM/YYYY)' in line:
        # print("coming here")
        company_info['Financial Year data to'] = all_lines[i + 2].strip()
    elif "Date of AGM (DD/MM/YYYY)" in line:
        company_info['DateOnWhichAGMIsHeld'] = all_lines[i + 2].strip()

    i += 1
# -----------------------------
# :two: Extract financial data
# -----------------------------
financial_data = []
i = 0
while i < len(all_lines):
    line = all_lines[i]
    if not is_number(line):  # heading lines
        heading_parts = [line]
        j = i + 1
        while j < len(all_lines) and not is_number(all_lines[j]):
            heading_parts.append(all_lines[j])
            j += 1
        heading = ' '.join(heading_parts)
        # Collect two numeric values if available
        if j + 1 < len(all_lines) and is_number(all_lines[j]) and is_number(all_lines[j + 1]):
            values = [float(all_lines[j].replace(',', '')), float(all_lines[j + 1].replace(',', ''))]
            financial_data.append({heading: values})
            i = j + 2
        else:
            i += 1
    else:
        i += 1
# -----------------------------
# :three: Combine and output JSON
# -----------------------------
output = {
    "company_info": company_info,
    "financial_data": financial_data
}
json_output = json.dumps(output, indent=4)
output['financial_data']


[{'(iv) *Pin Code/Zip Code Gurgaon Sector 45 (v)  Area/Locality Sector -45 (vi)  *City Gurgaon (vii) *District Haryana (viii) *State/UT (c) Particulars of the service provider (if any) Oracle India Private Limited (i) Name of the service provider IPV4 (ii) Internet protocol address of service provider Page 6 of 27 Gurugram (iii) Location of the service provider (iv) Whether books of account and other books and papers are maintained on cloud No Yes Oracle India Private Limited, 7th, 8th & 9th Floor, One Horizon Center, DLF Golf Course Road, DLF City V, Sector 43, Gurugram - 122003. Haryana. (v) Address as provided by the service provider Part B - Balance Sheet PART I - BALANCE SHEET Particulars Figures as at the end of (Current reporting period) (in Rs.) (DD/MM/YYYY) 31/03/2025 Figures as at the end of (Previous reporting period) (in Rs.) (DD/MM/YYYY) 31/03/2024 Reason for change in pre-filled figures of previous reporting period Figures as at the beginning of (Previous reporting period

In [43]:
############## testing #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import re

# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\CFS\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_05_09_2025.pdf"
input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\pdfs\CFS\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_26_08_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
reader = PdfReader(input_pdf_path)
writer = PdfWriter()

# -----------------------------
# Extract all lines from PDF
# -----------------------------
all_lines = []

with pdfplumber.open(input_pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        # print("page_num", page_num)
        print(page.images)
        # for line in page.extract_text_lines():
        #     print(line)

[{'x0': 269.68, 'y0': 689.41, 'x1': 354.27, 'y1': 774.0, 'width': 84.58999999999997, 'height': 84.59000000000003, 'name': 'img1', 'stream': <PDFStream(106): raw=7328, {'Length': 7328, 'BitsPerComponent': 8, 'ColorSpace': [/'CalRGB', <PDFObjRef:168>], 'Filter': /'FlateDecode', 'Height': 200, 'Intent': /'Perceptual', 'SMask': <PDFObjRef:105>, 'Subtype': /'Image', 'Type': /'XObject', 'Width': 200}>, 'srcsize': (200, 200), 'imagemask': None, 'bits': 8, 'colorspace': [/'CalRGB', {'Gamma': [2.2, 2.2, 2.2], 'Matrix': [0.41239, 0.21264, 0.01933, 0.35758, 0.71517, 0.11919, 0.18045, 0.07218, 0.9504], 'WhitePoint': [0.95043, 1, 1.09]}], 'mcid': None, 'tag': None, 'object_type': 'image', 'page_number': 1, 'top': 18.0, 'bottom': 102.59000000000003, 'doctop': 18.0}]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


In [40]:
############## Code to fetch text line info from a single  CFS pdf #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import re

# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\CFS\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_05_09_2025.pdf"
input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\pdfs\CFS\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_26_08_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
# input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
reader = PdfReader(input_pdf_path)
writer = PdfWriter()

# -----------------------------
# Extract all lines from PDF
# -----------------------------
all_lines = []

with pdfplumber.open(input_pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        # print("page_num", page_num)
        for line in page.extract_text_lines():
            # print(line['text'])
            all_lines.append(line['text'].replace("\xa0", " ").strip())



# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------
company_info = {}
i = 0
while i < len(all_lines):
    line = all_lines[i]
    # print(line)
    # print("*Corporate Identity Number (CIN)" in line)
    if '*Corporate Identity Number (CIN)' in line:
        company_info['cin'] = line.split("(CIN)")[-1].strip()
        if line.split("(CIN)")[-1].strip() == "":
            company_info['cin'] = all_lines[i + 1].strip()
    elif '*Name of the company' in line:
        company_info['name'] = line.split("company")[-1].strip()
        if line.split("company")[-1].strip() == "":
            name_parts = []
            j = i + 1
            while j < len(all_lines) and not all_lines[j].startswith('(b)'):
                name_parts.append(all_lines[j].strip())
                j += 1
            company_info['name'] = ' '.join(name_parts)
    elif '*Address of the registered office of the company' in line:
        addr_parts = []
        j = i + 1
        while j < len(all_lines) and not all_lines[j].startswith('(c)'):
            addr_parts.append(all_lines[j].strip())
            j += 1
        company_info['address'] = ', '.join(addr_parts)
    elif 'e-mail ID' in line:
        company_info['email'] = line.lower().split("company")[-1].strip()
        if line.lower().split("company")[-1].strip() == "":
            company_info['email'] = all_lines[i + 1].strip()
    elif '*From (DD/MM/YYYY)' in line:
        company_info['Financial Year data from'] = line.split("(DD/MM/YYYY)")[-1].strip()
        # print("coming here")
        if line.split("(DD/MM/YYYY)")[-1].strip() == "":
            company_info['Financial Year data from'] = all_lines[i + 1].strip()
    elif '*To (DD/MM/YYYY)' in line:
        company_info['Financial Year data to'] = line.split("(DD/MM/YYYY)")[-1].strip()
        # print("coming here")
        if line.split("(DD/MM/YYYY)")[-1].strip() == "":
            company_info['Financial Year data to'] = all_lines[i + 1].strip()
    elif "Date of AGM (DD/MM/YYYY)".lower() in line.lower() and "due" not in line.lower():
        company_info['DateOnWhichAGMIsHeld'] = line.split("(DD/MM/YYYY)")[-1].strip()
        if line.split("(DD/MM/YYYY)")[-1].strip() == "":
            company_info['DateOnWhichAGMIsHeld'] = all_lines[i + 1].strip()
    elif 'Nature of consolidated financial statements'.lower() in line.lower():
        # print("coming here")
        company_info['DateOnWhichAGMIsHeld'] = line.split("financial statements")[-1].strip()
        if line.split("financial statements")[-1].strip() == "":
            company_info['ReportNature'] = all_lines[i + 1].strip()
    elif 'Income-tax (PAN) of auditor or auditor’s firm'.lower() in line.lower():
        # print("coming here")
        company_info['AuditorFirmPAN'] = line.split("auditor’s firm")[-1].strip()
        if line.split("auditor’s firm")[-1].strip() == "":
            company_info['AuditorFirmPAN'] = all_lines[i + 1].strip()
    elif 'Name of the auditor or auditor’s firm'.lower() in line.lower():
        # print("coming here")
        company_info['NameOfAuditorFirm'] = line.split("auditor’s firm")[-1].strip()
        if line.split("auditor’s firm")[-1].strip() == "":        
            company_info['NameOfAuditorFirm'] = all_lines[i + 1].strip()
    elif 'Membership number of auditor or auditor’s firm’s registration'.lower() in line.lower():
        # print("coming here")
        company_info['AuditFirmRegNum'] = line.split("Membership number of auditor or auditor’s firm’s registration")[-1].strip()
        if line.split("Membership number of auditor or auditor’s firm’s registration")[-1].strip() == "":
            company_info['AuditFirmRegNum'] = all_lines[i + 1].strip()
    elif 'Address of the auditor or auditor’s firm'.lower() in line.lower():
        # print("coming here")
        address = ""
        address_line_1 = ""
        address_line_2 = ""
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "Address Line 1".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                address = all_lines[i+j].split("Address Line 1")[-1].strip()
                if address == "":
                    address = all_lines[i+j+1].strip()
                # print("coming here", address)
                break
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "Address Line 2".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                if "country" not in all_lines[i+j+1].lower():
                    address_line_2 = all_lines[i+j+1].strip()
                else:
                    address_line_2 = all_lines[i+j].split("Address Line 2")[-1].strip()
                address = address + " " + address_line_2
                break
        company_info['AddressOfAuditor1'] = address
        # print(all_lines[i+1])
        # if "Address Line 1".lower() in all_lines[i+1].lower():
        #     address = all_lines[i + 2].strip()
        # if "Address Line 2" in all_lines[i+3]:
        #     address_line_2 = all_lines[i + 4].strip()
        #     if "country" not in address_line_2.lower():
        #         address = address + " " + address_line_2
        # company_info['AddressOfAuditor1'] = address.strip()
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "Country".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                company_info['Auditor1AddressCountry'] = all_lines[i+j].lower().split("country")[-1].strip()
                if all_lines[i+j].lower().split("country")[-1].strip() == "":
                    company_info['Auditor1AddressCountry'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "pin code".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                company_info['Auditor1AddressPIN'] = all_lines[i+j].lower().split("pin code")[-1].strip()
                if all_lines[i+j].lower().split("pin code")[-1].strip() == "":
                    company_info['Auditor1AddressPIN'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "city".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                company_info['Auditor1AddressCity'] = all_lines[i+j].lower().split("city")[-1].strip()
                if all_lines[i+j].lower().split("city")[-1].strip() == "":
                    company_info['Auditor1AddressCity'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 10):
            # print("coming here", all_lines[i+j].lower())
            if "state".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                company_info['Auditor1AddressState'] = all_lines[i+j].lower().split("state")[-1].strip()
                if all_lines[i+j].lower().split("state")[-1].strip() == "":
                    company_info['Auditor1AddressState'] = all_lines[i+j+1].strip()
                break
    elif 'Details of the member signing for the above firm'.lower() in line.lower():
        for j in range(1, 5):
            # print("coming here", all_lines[i+j].lower())
            if "Name of the member".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                company_info['Auditor1MemberName'] = all_lines[i+j].split("member")[-1].strip()
                if all_lines[i+j].split("member")[-1].strip() == "":
                    company_info['Auditor1MemberName'] = all_lines[i+j+1].strip()
                break
        for j in range(1, 5):
            # print("coming here", all_lines[i+j].lower())
            if "Membership number".lower() in all_lines[i+j].lower():
                # print("coming here", all_lines[i+j+1].strip())
                company_info['Auditor1MemberNumber'] = all_lines[i+j].split("number")[-1].strip()
                if all_lines[i+j].split("number")[-1].strip() == "":
                    company_info['Auditor1MemberNumber'] = all_lines[i+j+1].strip()
                break


        

    i += 1
# -----------------------------
# :two: Extract financial data
# -----------------------------
financial_data = []
i = 0
while i < len(all_lines):
    line = all_lines[i]
    if not is_number(line):  # heading lines
        heading_parts = [line]
        j = i + 1
        while j < len(all_lines) and not is_number(all_lines[j]):
            heading_parts.append(all_lines[j])
            j += 1
        heading = ' '.join(heading_parts)
        # Collect two numeric values if available
        if j + 1 < len(all_lines) and is_number(all_lines[j]) and is_number(all_lines[j + 1]):
            values = [float(all_lines[j].replace(',', '')), float(all_lines[j + 1].replace(',', ''))]
            financial_data.append({heading: values})
            i = j + 2
        else:
            i += 1
    else:
        i += 1
# -----------------------------
# :three: Combine and output JSON
# -----------------------------
output = {
    "company_info": company_info,
    "financial_data": financial_data
}
json_output = json.dumps(output, indent=4)
output['company_info']


{'cin': 'U74899DL1991PLC046774',
 'name': 'HERO FINCORP LIMITED',
 'address': 'LOK VASANT VIHAR, NEW DELHI,, India, 110057',
 'email': '*****tors@herofincorp.com',
 'Financial Year data from': '01/04/2024',
 'Financial Year data to': '31/03/2025',
 'DateOnWhichAGMIsHeld': '28/07/2025',
 'ReportNature': 'Adopted Financial statements',
 'AuditorFirmPAN': 'AACFD4815A',
 'AuditFirmRegNum': 'number 117366W/W100018',
 'NameOfAuditorFirm': 'DELOITTE HASKINS AND SELLS LLP',
 'AddressOfAuditor1': '32nd Floor ',
 'Auditor1AddressCountry': 'india',
 'Auditor1AddressPIN': '400013',
 'Auditor1AddressCity': 'mumbai',
 'Auditor1MemberName': 'Mukesh Jain',
 'Auditor1MemberNumber': '108262'}

In [32]:
all_lines

['Form No. AOC-4 NBFC (Ind AS) 1-19402213823_SRN_FORM_1755767638492 Form language',
 'Form for filing financial statement and other English Hindi',
 'documents with the Registrar',
 '[Pursuant to section 137 of the Companies Act, 2013',
 'and sub-rule (1A) of Rule 12',
 'of Companies (Accounts) Rules, 2014]',
 '1755767638492',
 'Refer instruction kit for filing the form',
 'All fields marked in * are mandatory',
 'Figures appearing in the eForm should be entered in Absolute Rupees only. Figures should not be rounded off in any other unit like',
 'hundreds, thousands, lakhs, millions or crores',
 'SEGMENT- I: GENERAL INFORMATION OF THE COMPANY AND PARTICULARS IN RESPECT OF BALANCE SHEET',
 'Part A - General information of the Company',
 '1 (a) *Corporate Identity Number (CIN)',
 'L65921MH1991PLC059642',
 '(b) Authorised capital of the company as on the date of filing',
 '5500000000',
 '(c) Number of members of the company as on the date of filing',
 '2 (a) *Name of the company',
 'MAHIN

In [25]:
############## Code to fetch text line info from a multiple CFS pdfs #####################
import pdfplumber
from PyPDF2 import PdfWriter, PdfReader
import json
import os

# Helper: check if a string is a number
# Handles negative numbers, decimals, and commas
# -----------------------------
def is_number(s):
    try:
        float(s.replace(',', ''))
        return True
    except ValueError:
        return False
# -----------------------------
# :one: Extract company info
# -----------------------------


input_dir = "pdfs\\CFS"

for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        print("filename :", filename)
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_16_08_2025.pdf"
        # input_pdf_path =r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_16_08_2025.pdf"
        # input_pdf_path = r"D:\Nexensus_Projects\pdfFoms\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
        input_pdf_path = r"D:\\Nexensus_Projects\\pdfFoms\\pdfs\\CFS\\" + filename
        # reader = PdfReader(input_pdf_path)
        # writer = PdfWriter()

        # -----------------------------
        # Extract all lines from PDF
        # -----------------------------
        all_lines = []

        with pdfplumber.open(input_pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # print("page_num", page_num)
                for line in page.extract_text_lines():
                    # print(line['text'])
                    if "Part B - Balance Sheet" in line['text']:
                        print("starting page of balance sheet", page_num + 1)
                        # print(line['text'])
                        BS_starting_page = page_num + 1
                    if "TOTAL LIABILITIES" in line['text']:
                        print("ending page of balance sheet", page_num+1)
                        BS_ending_page = page_num + 1
                    all_lines.append(line['text'].replace("\xa0", " ").strip())




        company_info = {}
        i = 0
        while i < len(all_lines):
            line = all_lines[i]
            # print(line)
            # print("*Corporate Identity Number (CIN)" in line)
            if '*Corporate Identity Number (CIN)' in line:
                company_info['cin'] = line.split("(CIN)")[-1].strip()
                if line.split("(CIN)")[-1].strip() == "":
                    company_info['cin'] = all_lines[i + 1].strip()
            elif '*Name of the company' in line:
                # company_info['name'] = line.split("company")[-1].strip()
                # if line.split("company")[-1].strip() == "":
                name_parts = []
                j = i + 1
                print(all_lines[j])
                while j < len(all_lines) and not all_lines[j].startswith('(b)'):
                    name_parts.append(all_lines[j].strip())
                    j += 1
                company_info['name'] = line.split("company")[-1].strip() + " " + ' '.join(name_parts)
            elif '*Address of the registered office of the company' in line:
                addr_parts = []
                j = i + 1
                while j < len(all_lines) and not all_lines[j].startswith('(c)'):
                    addr_parts.append(all_lines[j].strip())
                    j += 1
                company_info['address'] = line.split("company")[-1].strip() + " " + ', '.join(addr_parts)
            elif 'e-mail ID' in line:
                # company_info['email'] = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', line)[0].strip()
                company_info['email'] = line.lower().split("company")[-1].strip()
                if line.lower().split("id")[-1].strip() == "":
                    company_info['email'] = all_lines[i + 1].strip()
            elif '*From (DD/MM/YYYY)' in line:
                company_info['Financial Year data from'] = line.split("(DD/MM/YYYY)")[-1].strip()
                # print("coming here")
                if line.split("(DD/MM/YYYY)")[-1].strip() == "":
                    company_info['Financial Year data from'] = all_lines[i + 1].strip()
            elif '*To (DD/MM/YYYY)' in line:
                company_info['Financial Year data to'] = line.split("(DD/MM/YYYY)")[-1].strip()
                # print("coming here")
                if line.split("(DD/MM/YYYY)")[-1].strip() == "":
                    company_info['Financial Year data to'] = all_lines[i + 1].strip()
            elif "Date of AGM (DD/MM/YYYY)".lower() in line.lower() and "due" not in line.lower():
                company_info['DateOnWhichAGMIsHeld'] = line.split("(DD/MM/YYYY)")[-1].strip()
                if line.split("(DD/MM/YYYY)")[-1].strip() == "":
                    company_info['DateOnWhichAGMIsHeld'] = all_lines[i + 1].strip()
            elif 'Nature of consolidated financial statements'.lower() in line.lower():
                # print("coming here")
                company_info['DateOnWhichAGMIsHeld'] = line.split("financial statements")[-1].strip()
                if line.split("financial statements")[-1].strip() == "":
                    company_info['ReportNature'] = all_lines[i + 1].strip()
            elif 'Income-tax (PAN) of auditor or auditor’s firm'.lower() in line.lower():
                # print("coming here")
                company_info['AuditorFirmPAN'] = line.split("auditor’s firm")[-1].strip()
                if line.split("auditor’s firm")[-1].strip() == "":
                    company_info['AuditorFirmPAN'] = all_lines[i + 1].strip()
            elif 'Name of the auditor or auditor’s firm'.lower() in line.lower():
                # print("coming here")
                company_info['NameOfAuditorFirm'] = line.split("auditor’s firm")[-1].strip()
                if line.split("auditor’s firm")[-1].strip() == "":        
                    company_info['NameOfAuditorFirm'] = all_lines[i + 1].strip()
            elif 'Membership number of auditor or auditor’s firm’s registration'.lower() in line.lower():
                # print("coming here")
                company_info['AuditFirmRegNum'] = line.split("registration number")[-1].strip()
                if line.split("registration number")[-1].strip() == "":
                    company_info['AuditFirmRegNum'] = all_lines[i + 1].strip()
            elif 'Address of the auditor or auditor’s firm'.lower() in line.lower():
                # print("coming here")
                address = ""
                address_line_1 = ""
                address_line_2 = ""
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "Address Line 1".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        address = all_lines[i+j].split("Address Line 1")[-1].strip()
                        if address == "":
                            address = all_lines[i+j+1].strip()
                        # print("coming here", address)
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "Address Line 2".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        if "country" not in all_lines[i+j+1].lower():
                            address_line_2 = all_lines[i+j+1].strip()
                        else:
                            address_line_2 = all_lines[i+j].split("Address Line 2")[-1].strip()
                        address = address + " " + address_line_2
                        break
                company_info['AddressOfAuditor1'] = address
                # print(all_lines[i+1])
                # if "Address Line 1".lower() in all_lines[i+1].lower():
                #     address = all_lines[i + 2].strip()
                # if "Address Line 2" in all_lines[i+3]:
                #     address_line_2 = all_lines[i + 4].strip()
                #     if "country" not in address_line_2.lower():
                #         address = address + " " + address_line_2
                # company_info['AddressOfAuditor1'] = address.strip()
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "Country".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['Auditor1AddressCountry'] = all_lines[i+j].lower().split("country")[-1].strip()
                        if all_lines[i+j].lower().split("country")[-1].strip() == "":
                            company_info['AuditorAddressCountry'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "pin code".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['Auditor1AddressPIN'] = all_lines[i+j].lower().split("pin code")[-1].strip()
                        if all_lines[i+j].lower().split("pin code")[-1].strip() == "":
                            company_info['AuditorAddressPIN'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "city".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['Auditor1AddressCity'] = all_lines[i+j].lower().split("city")[-1].strip()
                        if all_lines[i+j].lower().split("city")[-1].strip() == "":
                            company_info['AuditorAddressCity'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 10):
                    # print("coming here", all_lines[i+j].lower())
                    if "state".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['Auditor1AddressState'] = all_lines[i+j].lower().split("state")[-1].strip()
                        if all_lines[i+j].lower().split("state")[-1].strip() == "":
                            company_info['AuditorAddressState'] = all_lines[i+j+1].strip()
                        break
            elif 'Details of the member signing for the above firm'.lower() in line.lower():
                for j in range(1, 5):
                    # print("coming here", all_lines[i+j].lower())
                    if "Name of the member".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['Auditor1MemberName'] = all_lines[i+j].split("member")[-1].strip()
                        if all_lines[i+j].split("member")[-1].strip() == "":
                            company_info['AuditorMemberName'] = all_lines[i+j+1].strip()
                        break
                for j in range(1, 5):
                    # print("coming here", all_lines[i+j].lower())
                    if "Membership number".lower() in all_lines[i+j].lower():
                        # print("coming here", all_lines[i+j+1].strip())
                        company_info['Auditor1MemberNumber'] = all_lines[i+j].split("number")[-1].strip()
                        if all_lines[i+j].split("number")[-1].strip() == "":
                            company_info['Auditor1MemberNumber'] = all_lines[i+j+1].strip()
                        break

                
                

                
                
                

            i += 1
        # -----------------------------
        # :two: Extract financial data
        # -----------------------------
        financial_data = []
        i = 0
        while i < len(all_lines):
            line = all_lines[i]
            if not is_number(line):  # heading lines
                heading_parts = [line]
                j = i + 1
                while j < len(all_lines) and not is_number(all_lines[j]):
                    heading_parts.append(all_lines[j])
                    j += 1
                heading = ' '.join(heading_parts)
                # Collect two numeric values if available
                if j + 1 < len(all_lines) and is_number(all_lines[j]) and is_number(all_lines[j + 1]):
                    values = [float(all_lines[j].replace(',', '')), float(all_lines[j + 1].replace(',', ''))]
                    financial_data.append({heading: values})
                    i = j + 2
                else:
                    i += 1
            else:
                i += 1
        # -----------------------------
        # :three: Combine and output JSON
        # -----------------------------
        output = {
            "company_info": company_info,
            "financial_data": financial_data
        }
        json_output = json.dumps(output, indent=4)
        # print(output['company_info'])
        for key, value in output['company_info'].items():
            print(f"{key}: {value}")

        print("----------------------------------------------------------------")
        


filename : AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_05_09_2025.pdf
ending page of balance sheet 8
LIMITED
cin: U65191TN1994PLC079235
name: SMFG INDIA CREDIT COMPANY LIMITED Commerzone IT Park Tower B 1st Floor
address:  No 111, Mount Poonamallee Road,, Porur, Chennai, Sriperumbudur, India,, 600116
email: *****tarial@smfgindia.com
Financial Year data from: 01/04/2024
Financial Year data to: 31/03/2025
DateOnWhichAGMIsHeld: 07/08/2025
ReportNature: Adopted Financial statements
AuditorFirmPAN: AAAFW4298E
AuditFirmRegNum: 001076N/N500013
NameOfAuditorFirm: Walker Chandiok & Co LLP
AddressOfAuditor1: International Center, 
Auditor1AddressCountry: india
Auditor1AddressPIN: 400013
Auditor1AddressCity: mumbai
Auditor1MemberName: Manish Gujral
Auditor1MemberNumber: 105117
----------------------------------------------------------------
filename : AOC-4 CFS NBFC_Form AOC4 CFS NBFC Ind AS_12_09_2025.pdf
ending page of balance sheet 7
BOMBAY HOUSE 24 HOMI MODY
cin: U99999MH1917PTC000478
name: TATA

In [27]:
all_lines

['Form No. AOC-4 CFS NBFC (Ind AS)',
 '1-19096777486_SRN_FORM_1756201663706',
 'Form language',
 'Form for filing consolidated financial statements and',
 'other documents with the Registrar English Hindi',
 '[Pursuant to section 137 of the Companies Act, 2013 and',
 'sub-rule (1A) of Rule 12 of Companies (Accounts) Rules, 2014]',
 'Refer instruction kit for filing the form',
 '1756201663706',
 'All fields marked in * are mandatory',
 'Figures appearing in the e-Form should be entered in Absolute Rupees only. Figures should not be rounded off in any other unit like',
 'hundreds, thousands, lakhs, millions or crores',
 'SEGMENT- I: GENERAL INFORMATION OF THE COMPANY AND PARTICULARS IN RESPECT OF BALANCE SHEET',
 'Part A - General information of the Company',
 '1 SRN of form AOC-4 NBFC (Ind AS) filed by the company for its standalone financial',
 'statements',
 '2 *Corporate Identity Number (CIN) U74899DL1991PLC046774',
 '3 (a) *Name of the company HERO FINCORP LIMITED',
 '34, COMMUNITY 